# Preprocessing of NTCIR-1 Adhoc Dataset

Note: This preprocessing is designd for [geniie-lab](https://github.com/geniie-lab/geniie-lab) experiments and not necessarily suitable for other tasks.

## Data path
- Get a copy of the dataset (see [README.md](README.md)).
- We assume that the downloaded file has been uncompressed to the following path.

In [ ]:
import os
os.environ['DATA'] = "path to your data folder"

In [ ]:
!ls $DATA

## Preprocessing

### Corpus files

In [ ]:
!tar xvfz $DATA/MLIR.TGZ -C $DATA/

In [ ]:
!iconv -f EUC-JP -t UTF-8 -c $DATA/mlir/ntc1-j1 > $DATA/mlir/ntc1-j1.utf8

In [ ]:
# Number of documents
!grep "^<ACCN" $DATA/mlir/ntc1-j1.utf8 | wc -l

In [ ]:
import re
def docs_jsonl(in_file):
    out_file = in_file + '.jsonl'
    with open(in_file, 'r') as f:
        s = f.read()
        s = re.sub('<ABST.P>|</ABST.P>', '', s)
        s = re.sub(r'\\', r'\\\\', s)
        s = re.sub('"', '\\"', s)

        accn = re.findall('<ACCN.*?>(.*)</ACCN>', s)
        titl = re.findall('<TITL.*?>(.*)</TITL>', s)
        abst = re.findall('<ABST.*?>(.*)</ABST>', s)

    with open(out_file, 'w') as f:
        for i in range(len(accn)):
            # text = titl + " " + abst
            f.write(f'{{ "doc_id": "{accn[i]}", "text": "{titl[i]} {abst[i]}" }}\n')

In [ ]:
docs_jsonl(os.getenv('DATA') + '/mlir/ntc1-j1.utf8')

In [ ]:
!wc -l $DATA/mlir/ntc1-j1.utf8.jsonl

In [ ]:
!head $DATA/mlir/ntc1-j1.utf8.jsonl

### Topic files

In [ ]:
!tar xvfz $DATA/TOPICS.TGZ -C $DATA/

In [ ]:
!iconv -f EUC-JP -t UTF-8 -c $DATA/topics/topic0001-0030 > $DATA/topics/topic0001-0030.utf8
!iconv -f EUC-JP -t UTF-8 -c $DATA/topics/topic0031-0083 > $DATA/topics/topic0031-0083.utf8

In [ ]:
import re
def topics_jsonl(in_file):
    out_file = in_file + '.jsonl'
    with open(in_file, 'r') as f:
        s = f.read()
        qid = re.findall('<TOPIC q=([^>]+)>', s)
        title = re.findall('<TITLE>\n(.*)\n</TITLE>', s)
        desc = re.findall('<DESCRIPTION>\n(.*)\n</DESCRIPTION>', s)
        narr = re.findall('<NARRATIVE>\n(.*)\n</NARRATIVE>', s)
    with open(out_file, 'w') as f:
        for i in range(len(qid)):
            f.write(f'{{ "query_id": "{qid[i]}", "text": "{title[i]}", "description": "{desc[i]}", "narrative": "{narr[i]}" }}\n')

In [ ]:
topics_jsonl(os.getenv('DATA') + '/topics/topic0001-0030.utf8')
topics_jsonl(os.getenv('DATA') + '/topics/topic0031-0083.utf8')

In [ ]:
!cat $DATA/topics/topic0001-0030.utf8.jsonl $DATA/topics/topic0031-0083.utf8.jsonl > $DATA/topics/topic0001-0083.utf8.jsonl

In [ ]:
import re
def ntcir_to_trec(in_file):
    out_file = in_file + '.trec'

    with open(in_file, 'r', encoding='utf-8') as f:
        s = f.read()

        qid = re.findall(r'<TOPIC q=([^>]+)>', s)
        title = re.findall(r'<TITLE>\s*(.*?)\s*</TITLE>', s, re.DOTALL)
        desc = re.findall(r'<DESCRIPTION>\s*(.*?)\s*</DESCRIPTION>', s, re.DOTALL)
        narr = re.findall(r'<NARRATIVE>\s*(.*?)\s*</NARRATIVE>', s, re.DOTALL)

    with open(out_file, 'w', encoding='utf-8') as f:
        for i in range(len(qid)):
            f.write(
f"""<top>
<num> Number: {qid[i]}
<title> {title[i].strip()}

<desc> Description:
{desc[i].strip()}

<narr> Narrative:
{narr[i].strip()}
</top>

"""
            )

In [ ]:
ntcir_to_trec(os.getenv('DATA') + '/topics/topic0001-0030.utf8')
ntcir_to_trec(os.getenv('DATA') + '/topics/topic0031-0083.utf8')

In [ ]:
!cat $DATA/topics/topic0001-0030.utf8.trec $DATA/topics/topic0031-0083.utf8.trec > $DATA/topics/topic0001-0083.utf8.trec

In [ ]:
!ls $DATA/topics

### Qrel files
- This test collection provides graded relevance scores (A: Relevant, B: Partially Relevant, C: Not Relevant)
- We convert them as follows.
    - A: 2
    - B: 1
    - C: 0

In [ ]:
!iconv -f EUC-JP -t UTF-8 -c $DATA/mlir/rel2_ntc1-j1_0001-0030 > $DATA/mlir/rel2_ntc1-j1_0001-0030.utf8
!iconv -f EUC-JP -t UTF-8 -c $DATA/mlir/rel2_ntc1-j1_0031-0083 > $DATA/mlir/rel2_ntc1-j1_0031-0083.utf8

In [ ]:
def qrel_graded_tsv(in_file):
    out_file = in_file + '.tsv'
    with open(in_file, 'r') as f, open(out_file, 'w') as f2:
        for line in f:
            line = line.rstrip()
            flds = line.split('\t')
            if flds[1] == 'A':
                f2.write(f'{flds[0]}\tQ0\t{flds[2]}\t2\n')
            if flds[1] == 'B':
                f2.write(f'{flds[0]}\tQ0\t{flds[2]}\t1\n')
            if flds[1] == 'C':
                f2.write(f'{flds[0]}\tQ0\t{flds[2]}\t0\n')

In [ ]:
qrel_graded_tsv(os.getenv('DATA') + '/mlir/rel2_ntc1-j1_0001-0030.utf8')
qrel_graded_tsv(os.getenv('DATA') + '/mlir/rel2_ntc1-j1_0031-0083.utf8')

In [ ]:
!cat $DATA/mlir/rel2_ntc1-j1_0001-0030.utf8.tsv $DATA/mlir/rel2_ntc1-j1_0031-0083.utf8.tsv > $DATA/mlir/rel2_ntc1-j1_0001-0083.utf8.tsv

In [ ]:
!ls $DATA/mlir/

## Register to ir_datasets module locally

- Dataset name: `ntcir1-adhoc`

In [ ]:
# Remove old cache (if any)
!rm -rf ~/.ir_datasets/ntcir1-adhoc

In [ ]:
# Copy files
!mkdir -p ~/.ir_datasets/ntcir1-adhoc
!cp $DATA/mlir/ntc1-j1.utf8.jsonl ~/.ir_datasets/ntcir1-adhoc/
!cp $DATA/mlir/rel2_ntc1-j1_0001-0083.utf8.tsv ~/.ir_datasets/ntcir1-adhoc/
!cp $DATA/topics/topic0001-0083.utf8.trec ~/.ir_datasets/ntcir1-adhoc/

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas

In [ ]:
sys.path.append(os.path.join(os.getcwd()))

In [ ]:
import ir_datasets
import ntcir1_adhoc
dataset = ir_datasets.load('ntcir1-adhoc')
docstore = dataset.docs_store()
docstore.build()

In [ ]:
dataset.docs_cls().__annotations__

In [ ]:
docstore = dataset.docs_store()
docstore.get('gakkai-0000011144').text # the one in the overview paper

In [ ]:
dataset.queries_cls().__annotations__

In [ ]:
import pandas as pd
pd.DataFrame(dataset.queries_iter())

In [ ]:
dataset.qrels_defs()

In [ ]:
pd.DataFrame(dataset.qrels_iter())